# Figure 6 and Extended Data 9 — variant effects at protein–protein interfaces

For every complex containing one of the 17 profiled proteins — experimental
(RCSB PDB), AlphaFold-Multimer (Predictomes, SPOC ≥ 0.69) and RoseTTAFold2-PPI
(Zhang et al. 2025) — do missense variants at the interface score differently
from matched surface controls on the same protein?

The notebook runs the analysis in five steps, and draws each panel where the
step it depends on is done. The drawing code is in `interactions/ppi_utils.py`.

| Section | Analysis | Panels |
|---|---|---|
| A | one interface-vs-control test per complex | |
| B | the significance rule: each complex tested on its own | 6c, 6d |
| C | distinct protein pairs | 6a, ED9a, ED9b |
| D | worked interfaces: the BRAF dimer, CRAF on HSP90–CDC37, and BRAF–ITCH | 6b, ED9c–h |
| E | where on each protein the partners bind: interface clusters | ED9i–k |

**Inputs.** Scores from `output/annotated_combined.tsv` (see `Annotations.ipynb`),
and the structure sets in `data/interactions/` — the structures this analysis
was run on, frozen and deposited with the data rather than re-fetched.

**Outputs.** `output/interactions/` (per-complex statistics) and
`output/figures/fig6/`.

**Environment.** `interactions/environment.yaml` (`labelseq_mapk_interactions`):
this code is pinned to pandas 1.5. Run the notebook in that kernel.

In [ ]:
# ── Optional: run from the Zenodo deposit instead of the raw data ──────────────
# Normally every input is built from the barcode counts (Scoring.ipynb ->
# Annotations.ipynb). To run the figure notebooks from the deposited tables
# instead, uncomment the line below and run this cell once; it fills data/ and
# output/ for all six figure notebooks, so the rest of this notebook runs
# unchanged. Zenodo: https://doi.org/10.5281/zenodo.ZENODO_RECORD_ID
# All of Us data are not in the deposit (Controlled Tier; per-variant data may
# not be redistributed), so the All of Us rows of Fig. 4f and Extended Data
# Fig. 7e come out empty; every other panel reproduces.
#
# !python scripts/fetch_zenodo_data.py --record ZENODO_RECORD_ID

In [ ]:
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')          # panels are saved, then shown from the file
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sys.path.insert(0, 'interactions')
from ppi_utils import (
    # loading, pooling, the significance rule
    load_combined_results, aggregate_volcano_data, prepare_volcano_data,
    flag_significant, parse_positions,
    # interface footprints and clusters
    build_footprints, representative_cbeta_coords,
    cluster_sites, disp, site_category_2, plot_sites_bar_compact,
    DIST_TOL, MIN_FOOTPRINT, SITE_CUTOFF,
    SITE_CATS_2, SITE_CAT_COLOR_2, SITE_CAT_DESC_2,
    # BRAF dimer categories
    REAL, RCTRL, ALT, ACTRL, REAL_ID, ALT_ID, _cohens_d,
    IFACE_SHADES, CTRL_COLOR_REAL, CTRL_COLOR_ALT,
    # structure renders
    render_structure, plot_structure_panel, residue_scores, score_scale,
    plddt_keep, legend_patch, legend_dot, resolve_pdb_path, darken, plot_structure_key,
    STRUCTURE_VIEWS, PARTNER_COLOR,
    _paper_panel_data, select_representative_structures, surface_residues,
    render_surface, plot_surface_views, FAMILY_PALETTE, SHARED_TIERS, shared_tier,
    # drawing
    FIG6, plot_funnel, plot_volcanoes, plot_volcano_size_key, plot_source_venn,
    plot_interactions_by_class, plot_interface_violins, ALLD, ALL_COLOR, CAT_COLOR, substituted_q,
    plot_sites_bar, plot_iou_heatmaps,
)

FIG6.mkdir(parents=True, exist_ok=True)
with open('config/visualization_config.yaml') as fh:
    cfg = yaml.safe_load(fh)
RESULT_DIRS = dict(predictomes_dir=cfg['predictomes_dir'],
                   zhangetal_dir=cfg['zhangetal_dir'],
                   pdb_dir=cfg['pdb_output_dir'])
ASSAYS = ['abundance', 'activity']


def show(png, width=700):
    """Display a saved panel (a downscaled preview; the PDF beside it is the
    figure file)."""
    import io
    from PIL import Image as PILImage
    im = PILImage.open(png)
    im.thumbnail((2 * width, 2 * width))
    buf = io.BytesIO()
    im.save(buf, format='PNG', optimize=True)
    display(Image(data=buf.getvalue(), width=width))


def run(script, *args):
    """Run one of the scripts in interactions/ with this kernel's python."""
    subprocess.run([sys.executable, script, *args], check=True)


def pair_of(df):
    """Unordered protein pair per row: SOS1 scored against KRAS and KRAS scored
    against SOS1 are the same pair."""
    return pd.Series([frozenset(p) for p in zip(df['scored_protein_name'],
                                                df['interactor_name'])],
                     index=df.index)

## From structures to per-complex statistics

These steps are scripts in `interactions/` and are not re-run by default.

1. **Structure selection** — done once; the result is frozen in `data/interactions/`.
   ```
   # RCSB PDB heterocomplexes, SIFTS-renumbered, one representative pair per entity
   python interactions/fetch_pdb_controls.py \
       --uniprot-ids O14807,P00533,P01116,P04049,P04626,P07949,P08581,P10398,P15056,P36507,P62993,Q02750,Q06124,Q07889,Q07890,Q6VAB6,Q8IVT5 \
       --output-dir data/interactions/pdb_controls/ \
       --resolution-cutoff 3.5 --em-resolution-cutoff 4.5 \
       --experimental-methods "X-RAY DIFFRACTION,ELECTRON MICROSCOPY" \
       --min-sifts-coverage 0.5 --min-partner-length 10 --max-per-pair 5 --threads 20 \
       --extra-pdb-ids 4MNE,4QSY,3KUC,9AXM,6EPL,6VJJ,2Y4I,7JUZ,7JUW,1MW4 \
       --uniprot-aliases P29678:Q02750
   # homodimer pairs the entity-level selection cannot reach (4MNE, 3NY5, 6UAN)
   python interactions/add_homodimer_controls.py
   # Predictomes AlphaFold-Multimer models; the analysis uses SPOC >= 0.69
   python interactions/extract_predictomes.py --uniprot-ids <the 17 above> \
       --predictomes-csv <predictomes pair scores> --tar-dir <predictomes tars> \
       --output-dir data/interactions/predictomes_MAPK/ --spoc-threshold 0.5
   ```
   `matching_interactions_spoc069.csv` is `matching_interactions.csv` filtered to
   `spoc_score >= 0.69`. The Zhang et al. set is the models of every
   confident-pair prediction involving a profiled protein, with its pair table.
2. **Interface vs. control statistics** — `main.py`, once per source (20–40 min
   each on 10 cores; `interactions/run_main.sge` submits them to SGE).
3. **Inter-chain clash cache** — `compute_interchain_clashes.py`.

In [ ]:
RUN_MAIN = False       # ~1 h in total; or submit interactions/run_main.sge
RUN_CLASHES = False

if RUN_MAIN:
    for c in ['config/config_pdb_ctrl_domains_rsa25_gt10A.yaml',
              'config/config_predictomes_dom_r25_gt10.yaml',
              'config/config_zhang_dom_r25_gt10.yaml']:
        run('interactions/main.py', '--config', c)
if RUN_CLASHES:
    run('interactions/compute_interchain_clashes.py')

## A. One test per complex

For each complex and each profiled protein in it, `main.py` compared the
WT-relative scores of missense variants at **interface** positions (Cβ within
6 Å of the partner chain) with those at matched **control** positions on the same
protein — solvent-exposed (RSA > 0.25), inside an annotated domain, and ≥ 10 Å
from the interface — giving a Mann–Whitney p and a Cohen's d per assay. A complex
needs ≥ 3 scored interface and ≥ 3 scored control positions, and AlphaFold models
drop residues with pLDDT < 60. Complexes with a confident inter-chain Cβ clash
(< 3 Å) are removed as physically implausible.

In [ ]:
results = load_combined_results(**RESULT_DIRS)        # tested complexes, clashes removed
tested = results.dropna(subset=['log_fold_change', 'pval_fdr', 'cohens_d'])
tested = tested[tested['assay'].isin(ASSAYS)]

tested.groupby('predictor').agg(complexes=('structure', 'nunique'),
                                scored_proteins=('scored_protein_name', 'nunique'),
                                partners=('interactor_name', 'nunique'))

## B. The significance rule: each complex tested on its own

Every complex is its own test. BH-FDR is applied per assay across all complexes,
all sources together; a complex with both proteins profiled is tested once per
scored protein. A complex is **functionally supported** when its FDR-corrected
Mann–Whitney U test q < 0.05 and |Cohen's d| ≥ 0.2 (`flag_significant`), and a
protein pair is supported when any of its complexes is. Every count in the text
and every panel uses this rule; only the volcano plots merge complexes, for
display.

In [ ]:
complexes = tested.copy()
complexes['pval_fdr'] = np.nan
for assay in ASSAYS:
    rows = (complexes['assay'] == assay) & complexes['pval_mannwhitney'].notna()
    p = complexes.loc[rows, 'pval_mannwhitney'].clip(lower=1e-300)
    complexes.loc[rows, 'pval_fdr'] = multipletests(p, method='fdr_bh')[1]
complexes['neg_log_pval'] = -np.log10(complexes['pval_fdr'].clip(lower=1e-300))
complexes['supported'] = flag_significant(complexes)
complexes['pair'] = pair_of(complexes)

complexes.groupby(['assay', 'predictor'])['supported'].agg(tested='size', supported='sum')

### 6c and 6d — activity and abundance volcanoes

For display only, complexes of the same pair from the same source whose
interfaces overlap (IoU ≥ 0.3, one binding mode) are merged into one point: the
median Cohen's d of its complexes, and the BH-FDR q (per assay, across points) of
their geometric-mean p. Points are coloured by source; marker area is the median
number of interface residues. The corner counts are the supported complexes from
above, and the distinct protein pairs among them, by direction of effect.
Interactions discussed in the text are labelled, with a line to each of their points.

In [ ]:
corner_counts = {}
for assay in ASSAYS:
    sig = complexes[(complexes['assay'] == assay) & complexes['supported']]
    down, up = sig[sig['cohens_d'] <= -0.2], sig[sig['cohens_d'] >= 0.2]
    corner_counts[assay] = {'down': (len(down), down['pair'].nunique()),
                            'up': (len(up), up['pair'].nunique())}
display(pd.DataFrame({a: {'decreased: complexes': c['down'][0], 'decreased: pairs': c['down'][1],
                          'increased: complexes': c['up'][0], 'increased: pairs': c['up'][1]}
                      for a, c in corner_counts.items()}))

# The volcano points: complexes merged by binding mode (display only).
points = aggregate_volcano_data(tested)

# Interactions featured in the text: (scored protein, partner, binding modes or
# None for all, label). A parenthesis starts the label's second line. SOS1 binds
# KRAS at its catalytic site (CDC25 domain) and at an allosteric site (REM +
# CDC25), which the models place as separate binding modes; the same models give
# the two KRAS-side modes. 7K0V's two protomers are the BRAF-BRAF modes #2, #3.
FEATURED = [
    ('SOS1', 'KRAS', ['RasGEF'], 'SOS1–KRAS (catalytic)'),
    ('SOS1', 'KRAS', ['RasGEF+RasGEFN'], 'SOS1–KRAS (allosteric)'),
    ('KRAS', 'SOS1', ['H_N_K_Ras_like'], 'KRAS–SOS1 (catalytic)'),
    ('KRAS', 'SOS1', ['H_N_K_Ras_like #2'], 'KRAS–SOS1 (allosteric)'),
    ('BRAF', 'BRAF', ['STKc_Raf'], 'BRAF–BRAF (4MNE dimer)'),
    ('BRAF', 'BRAF', ['STKc_Raf #2', 'STKc_Raf #3'], 'BRAF–BRAF (crystal artifact)'),
    ('BRAF', 'ITCH', None, 'BRAF–ITCH'),
    ('RAF1', 'CDC37', None, 'CRAF–CDC37'),
    ('RAF1', 'HSP90AB1', None, 'CRAF–HSP90AB1'),
    ('KRAS', 'LZTR1', None, 'KRAS–LZTR1'),
]
points['volcano_label'] = None
for scored, partner, modes, label in FEATURED:
    rows = (points['scored_protein_name'] == scored) & (points['interactor_name'] == partner)
    if modes is not None:
        rows &= points['binding_mode'].isin(modes)
    assert set(points.loc[rows, 'assay']) == set(ASSAYS), label
    points.loc[rows, 'volcano_label'] = label.replace(' (', '\n(')

plot_volcanoes(prepare_volcano_data(points), corner_counts, None)
plot_volcano_size_key(points)
show(FIG6 / 'volcano_pub_activity_all_predictors.png')
show(FIG6 / 'volcano_pub_abundance_all_predictors.png')
show(FIG6 / 'volcano_size_legend.png', width=120)

## C. Distinct protein pairs

An interaction here is an unordered protein pair: a pair scored from both sides
counts once, and it is supported if any of its complexes is supported, in either
assay. A pair with both experimental and predicted structures is also counted by
class: supported **by** an experimental complex, or **by** a predicted one.

In [ ]:
tested_pairs = pair_of(tested)
experimental = complexes['predictor'] == 'RCSB_PDB'


def pairs_supported(rows):
    return rows.groupby('pair')['supported'].any()


per_pair = pd.DataFrame({
    'supported': pairs_supported(complexes),
    'activity': pairs_supported(complexes[complexes['assay'] == 'activity']),
    'abundance': pairs_supported(complexes[complexes['assay'] == 'abundance']),
    'has_experimental': complexes.groupby('pair')['predictor'].apply(lambda s: (s == 'RCSB_PDB').any()),
    'has_predicted': complexes.groupby('pair')['predictor'].apply(lambda s: (s != 'RCSB_PDB').any()),
})
for cls, rows in [('experimental', complexes[experimental]), ('predicted', complexes[~experimental])]:
    for assay in ['activity', 'abundance']:
        per_pair[f'{assay}_by_{cls}'] = pairs_supported(rows[rows['assay'] == assay])
    per_pair[f'supported_by_{cls}'] = pairs_supported(rows)
per_pair = per_pair.fillna(False)

n = len(per_pair)
exp_pairs, pred_pairs = per_pair[per_pair['has_experimental']], per_pair[per_pair['has_predicted']]
print(f"{n} distinct pairs tested; {per_pair['supported'].sum()} supported "
      f"(activity {per_pair['activity'].sum()}, abundance {per_pair['abundance'].sum()})")
print(f"{len(exp_pairs)} pairs with an experimental structure: {exp_pairs['supported_by_experimental'].sum()} "
      f"supported by it (activity {exp_pairs['activity_by_experimental'].sum()}, "
      f"abundance {exp_pairs['abundance_by_experimental'].sum()})")
print(f"{len(pred_pairs)} pairs with a predicted structure: {pred_pairs['supported_by_predicted'].sum()} "
      f"supported by it (activity {pred_pairs['activity_by_predicted'].sum()}, "
      f"abundance {pred_pairs['abundance_by_predicted'].sum()})")
supported = per_pair[per_pair['supported']]
print(f"{(~supported['has_experimental']).sum()} of the {len(supported)} supported pairs "
      f"have no experimental structure")

### Supplementary Table 3 — every complex

One row per complex and scored protein: a heterocomplex with both proteins
profiled contributes two rows, and a complex that was never tested (too few
interface or control positions, or an inter-chain clash) keeps its row with the
reason and no statistics. Activity and abundance Cohen's *d* and q sit side by
side, with the pair-level annotations repeated on every row of that pair.

Two annotations are merged from `interactions/pair_annotations.tsv`. Whether the
pair is reported in a curated interaction database (BioGRID multi-validated
physical interactions, UniProt curated "Interacts with"), which is recorded for
every pair. And partner druggability (Open Targets tractability, drugs and
chemical probes, with every molecule's target checked by hand against the
primary literature), which was curated for the supported pairs only and is blank
elsewhere. Both need network access, so they are versioned rather than rebuilt
here. Written as one sheet, `.xlsx` and
`.csv`.

In [ ]:
SUPP3 = FIG6 / 'supplementary_table_3_interactions'
SOURCE_NAME = {'RCSB_PDB': 'PDB', 'Predictomes': 'AF-M', 'Zhang_et_al': 'RF2-PPI'}
KEY = ['structure', 'predictor', 'scored_protein_name', 'interactor_name', 'scored_chain']


def pair_key(p):
    """'A|B', sorted; a homodimer is its own partner, so it becomes 'A|A'."""
    g = sorted(p)
    return '|'.join(g if len(g) == 2 else g * 2)


# Every complex the pipeline saw, including the ones it could not test.
every = load_combined_results(**RESULT_DIRS, include_skipped=True, apply_clash_filter=False)
rows = every.drop_duplicates(KEY)[KEY + ['skipped', 'skip_reason']].copy()

# Per assay: the statistics, taken from `complexes` so the q-values are the same
# BH-corrected ones the figures use.
stats = complexes.set_index(KEY)
for assay in ASSAYS:
    a = stats[stats['assay'] == assay][['n_interface_positions', 'n_control_positions',
                                        'cohens_d', 'pval_fdr', 'supported']]
    rows = rows.join(a.add_prefix(f'{assay}_'), on=KEY)

tested_keys = set(map(tuple, complexes[KEY].drop_duplicates().to_numpy()))
is_tested = [tuple(t) in tested_keys for t in rows[KEY].to_numpy()]
rows['status'] = np.where(
    is_tested, 'assessable',
    np.where(rows['skipped'].fillna(False), 'not assessable: ' + rows['skip_reason'].astype(str),
             'not assessable: inter-chain clash'))
rows['structure_source'] = rows['predictor'].map(SOURCE_NAME)
rows['pair'] = [pair_key(frozenset((a, b)))
                for a, b in zip(rows['scored_protein_name'], rows['interactor_name'])]

# Pair-level annotations, repeated on every complex of that pair.
# The annotation file lists every pair of a scored protein reported in BioGRID
# multi-validated or UniProt curated, which is a superset of the pairs modelled
# here; a pair absent from it is genuinely unreported, hence fillna(False).
ann = pd.read_csv('interactions/pair_annotations.tsv', sep='\t')
supported_pairs = {pair_key(p) for p in per_pair.index[per_pair['supported']]}
rows = rows.merge(ann, left_on='pair', right_on='pair_key', how='left').drop(columns='pair_key')
for col in ['biogrid_multivalidated', 'uniprot_curated', 'reported_in_curated_database']:
    rows[col] = rows[col].fillna(False).astype(bool)
rows['pair_functionally_supported'] = rows['pair'].isin(supported_pairs)
# Druggability was curated for the supported pairs only, so it is left blank
# elsewhere rather than recorded as absent.
rows['partner_has_verified_molecule'] = np.where(
    rows['pair_functionally_supported'], rows['verified_partner'].notna(), None)

supp3 = rows[['pair', 'scored_protein_name', 'interactor_name', 'scored_chain',
              'structure', 'structure_source', 'status',
              'activity_n_interface_positions', 'activity_n_control_positions',
              'activity_cohens_d', 'activity_pval_fdr', 'activity_supported',
              'abundance_cohens_d', 'abundance_pval_fdr', 'abundance_supported',
              'pair_functionally_supported', 'reported_in_curated_database',
              'biogrid_multivalidated', 'uniprot_curated', 'partner_has_verified_molecule',
              'verified_partner', 'partner_sm_tractability', 'partner_n_drugs',
              'partner_top_drugs', 'partner_chemical_probes',
              'partner_druggability_verified', 'partner_verified_molecule',
              'partner_druggability_reference']].rename(
    columns={'scored_protein_name': 'scored_protein', 'interactor_name': 'partner',
             'activity_n_interface_positions': 'n_interface_positions',
             'activity_n_control_positions': 'n_control_positions',
             'activity_pval_fdr': 'activity_q', 'abundance_pval_fdr': 'abundance_q'})
# True / False / blank rather than 1.0 / 0.0 / NaN, and whole position counts.
for col in ['activity_supported', 'abundance_supported', 'pair_functionally_supported',
            'partner_has_verified_molecule']:
    supp3[col] = supp3[col].astype('boolean')
for col in ['n_interface_positions', 'n_control_positions', 'partner_n_drugs']:
    supp3[col] = supp3[col].astype('Int64')
supp3 = supp3.sort_values(['pair', 'scored_protein', 'structure']).reset_index(drop=True)
supp3.to_csv(SUPP3.with_suffix('.csv'), index=False)
supp3.to_excel(SUPP3.with_suffix('.xlsx'), sheet_name='Supplementary Table 3', index=False)

ok = supp3['status'] == 'assessable'
print(f'{len(supp3)} rows, {len(supp3.columns)} columns -> {SUPP3.name}.xlsx / .csv')
print(f"  {supp3['structure'].nunique()} complexes, {supp3['pair'].nunique()} pairs")
print(f"  {ok.sum()} assessable rows ({supp3.loc[ok, 'structure'].nunique()} complexes), "
      f"{(~ok).sum()} not assessable")
print(f"  {supp3.loc[ok, 'pair'].nunique()} pairs assessable, "
      f"{supp3.loc[supp3['pair_functionally_supported'], 'pair'].nunique()} supported")
sup_pairs = supp3[supp3['pair_functionally_supported']].drop_duplicates('pair')
print(f"  of the supported pairs, {int(sup_pairs['reported_in_curated_database'].sum())} are "
      f"reported in a curated database and "
      f"{int(sup_pairs['partner_has_verified_molecule'].sum())} have a verified partner molecule")
supp3.head()

### 6a — from complexes to functionally supported interactions

Complexes per source as fetched, the complexes left after the clash filter and
the ≥ 3-position requirement (assessable), and the distinct pairs among them.

In [ ]:
FUNNEL_LABEL = {'RCSB_PDB': 'PDB', 'Predictomes': 'AlphaFold', 'Zhang_et_al': 'RF2-PPI'}


def complexes_per_source(df):
    unique = df.drop_duplicates('structure')
    return {lab: int((unique['predictor'] == p).sum()) for p, lab in FUNNEL_LABEL.items()}


all_complexes = load_combined_results(**RESULT_DIRS, include_skipped=True,
                                      apply_clash_filter=False)
F = dict(raw=complexes_per_source(all_complexes),
         assessable=complexes_per_source(tested),
         tested_union=len(per_pair),
         sig_union=int(per_pair['supported'].sum()))
display(pd.DataFrame({'complexes': F['raw'], 'assessable': F['assessable']}))

plot_funnel(F)
show(FIG6 / 'pipeline_funnel.png', width=300)

### ED9a — pairs by structure source

A pair is in a source's circle if that source contributes at least one assessable
complex; the mean number of complexes per pair is given for each source.

In [ ]:
VENN_LABEL = {'RCSB_PDB': 'PDB', 'Predictomes': 'AF-M', 'Zhang_et_al': 'RF2-PPI'}
pairs = {lab: set(tested_pairs[tested['predictor'] == p]) for p, lab in VENN_LABEL.items()}
P, A, R = pairs['PDB'], pairs['AF-M'], pairs['RF2-PPI']
subsets = (len(P - A - R), len(A - P - R), len((P & A) - R),
           len(R - P - A), len((P & R) - A), len((A & R) - P), len(P & A & R))
avg = {lab: tested.loc[tested['predictor'] == p, 'structure'].nunique() / len(pairs[lab])
       for p, lab in VENN_LABEL.items()}
print(dict(zip(['PDB only', 'AF-M only', 'PDB & AF-M', 'RF2-PPI only', 'PDB & RF2-PPI',
                'AF-M & RF2-PPI', 'all three'], subsets)))

plot_source_venn(subsets, avg, len(P | A | R))
show(FIG6 / 'source_pair_venn.png', width=450)

### ED9b — partners per protein, by source

Partners of each profiled protein, counted once per source that models the pair;
solid where at least one of that source's complexes of the pair is supported,
pale otherwise.

In [ ]:
SOURCE = {'RCSB_PDB': 'PDB', 'Predictomes': 'AF', 'Zhang_et_al': 'RF'}
partners = (complexes.assign(source=complexes['predictor'].map(SOURCE))
                     .groupby(['scored_protein_name', 'source', 'interactor'])['supported']
                     .any().reset_index())
per_protein = partners.groupby(['scored_protein_name', 'source']).agg(
    n_total=('interactor', 'nunique'), n_sig=('supported', 'sum'))
display(per_protein['n_total'].unstack(fill_value=0).assign(
    supported=per_protein['n_sig'].groupby(level=0).sum()).rename(index=disp))

plot_interactions_by_class(per_protein['n_total'].to_dict(), per_protein['n_sig'].to_dict())
show(FIG6 / 'interactor_class_summary.png')

## D. Worked interfaces

### The BRAF dimer: the side-to-side signalling dimer (4MNE) vs. a crystal contact (7K0V)

Each homodimer contributes one interface set — the union of both protomers'
contact residues — and one control set, the union of both protomers' controls
minus that interface. Each contact is tested against its own controls; nothing is
pooled across structures. The q-values are BH across all complexes per assay
(section B), with each structure's two protomer tests replaced by its pooled
test, plus the direct 4MNE-vs-7K0V comparison. 4MNE is the dimer with MEK1 bound, added to the PDB set
by `add_homodimer_controls.py`.

In [ ]:
braf_dimer = results[(results['scored_protein_name'] == 'BRAF')
                     & (results['interactor_name'] == 'BRAF')
                     & results['interface_positions'].notna()]
braf_pos = {}
for structure_id, iface_cat, ctrl_cat in [(REAL_ID, REAL, RCTRL), (ALT_ID, ALT, ACTRL)]:
    rows = braf_dimer[braf_dimer['structure'].str.contains(structure_id)]
    iface = set().union(*map(parse_positions, rows['interface_positions']))
    ctrl = set().union(*map(parse_positions, rows['control_positions'].dropna()))
    braf_pos[iface_cat], braf_pos[ctrl_cat] = iface, ctrl - iface


def library_scores(libraries):
    """One protein's scores, with integer positions."""
    s = scores[scores['library'].isin(libraries)].copy()
    s['Position'] = pd.to_numeric(s['Position'], errors='coerce')
    s = s.dropna(subset=['Position'])
    s['Position'] = s['Position'].astype(int)
    return s


def interface_tests(s, pos, comparisons):
    """Mann-Whitney p and Cohen's d for each (interface, control) comparison."""
    out = {}
    for assay in ['activity', 'abundance']:
        sub = s[(s['assay'] == assay) & (s['assay_treatment'] == 'No_treatment')
                & (s['Mutation Type'] == 'missense')].dropna(subset=['average score'])
        for a, b in comparisons:
            x = sub[sub['Position'].isin(pos[a])]['average score'].values
            y = sub[sub['Position'].isin(pos[b])]['average score'].values
            out[(assay, a.replace(chr(10), ' '), b.replace(chr(10), ' '))] = dict(
                n=f'{len(x)} vs {len(y)}', cohens_d=round(_cohens_d(x, y), 2),
                p=mannwhitneyu(x, y, alternative='two-sided').pvalue)
    return pd.DataFrame(out).T


# Read through the shared loader: the table carries Supplementary Table 2's
# names, and load_scores maps them back to the ones used here.
sys.path.insert(0, 'src')
from labelseq_mapk.figure_data import load_scores
scores = load_scores(usecols=['library', 'Position', 'assay', 'assay_treatment',
                              'Mutation Type', 'average score'])
braf = library_scores(['braf_cterm', 'braf_nterm'])
BRAF_PAIRS = [(REAL, RCTRL), (ALT, ACTRL), (REAL, ALT)]
braf_tests = interface_tests(braf, braf_pos, BRAF_PAIRS)

# q: BH per assay over all complexes, with the 4MNE and 7K0V protomer tests
# replaced by these pooled tests (and the direct 4MNE-vs-7K0V comparison added).
label = lambda c: c.replace('\n', ' ')
braf_q = {assay: substituted_q(
              complexes, assay,
              lambda r: ((r['scored_protein_name'] == 'BRAF') & (r['interactor_name'] == 'BRAF')
                         & r['structure'].str.contains(f'{REAL_ID}|{ALT_ID}')),
              {pair: braf_tests.loc[(assay, label(pair[0]), label(pair[1])), 'p']
               for pair in BRAF_PAIRS})
          for assay in ASSAYS}
braf_tests['q'] = [braf_q[a][next(pr for pr in BRAF_PAIRS if (label(pr[0]), label(pr[1])) == (i, c))]
                   for a, i, c in braf_tests.index]
braf_tests.assign(p=braf_tests['p'].map('{:.1e}'.format), q=braf_tests['q'].map('{:.1e}'.format))

### 6b and ED9c — the two dimers coloured by mean missense score

Both protomers coloured per residue (6b: 4MNE; ED9c: 7K0V); spheres mark the
residues analysed, interface and control. Rendered with `render_structure`.

In [ ]:
BRAF_DIMERS = [  # (title, structure, file, control-sphere colour)
    ('Side-to-side dimer, R509 (4MNE)', REAL_ID,
     'data/interactions/pdb_controls/P15056_P15056_4MNE_BC.pdb', CTRL_COLOR_REAL),
    ('Alternate contact (7K0V)', ALT_ID,
     'data/interactions/pdb_controls/P15056_P15056_7K0V_CA.pdb', CTRL_COLOR_ALT)]


def per_chain(rows, column):
    """{scored chain: positions} for one structure: each protomer keeps its own
    interface and its own controls."""
    return {ch: set().union(*map(parse_positions, vals.dropna()))
            for ch, vals in rows.groupby('scored_chain')[column]}


for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['braf_cterm', 'braf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre, cap=0.85 if assay == 'activity' else None)
    renders = []
    for title, key, pdb, ctrl_color in BRAF_DIMERS:
        rows = braf_dimer[braf_dimer['structure'].str.contains(key)]
        iface = per_chain(rows, 'interface_positions')
        ctrl = per_chain(rows, 'control_positions')
        spheres = ([(ch, p, IFACE_SHADES[key][ch], 0.85) for ch, p in iface.items()]
                   + [(ch, p, ctrl_color, 0.42) for ch, p in ctrl.items()])
        renders.append(render_structure(
            pdb, FIG6 / f'_braf_braf_{key}_{assay}_withspheres_render.png',
            scored_chains=['A', 'B'], scores=pos, centre=centre, vmin=vmin, vmax=vmax,
            sticks=iface, spheres=spheres, view=STRUCTURE_VIEWS[key], size=(2400, 2200)))
    plot_structure_panel(
        renders, [d[0] for d in BRAF_DIMERS], FIG6 / f'braf_braf_{assay}_withspheres',
        centre=centre, vmin=vmin, vmax=vmax, cbar_label=f'BRAF {assay}',
        legend=[legend_dot(IFACE_SHADES['4MNE']['B'], '4MNE interface, chain B'),
                legend_dot(IFACE_SHADES['4MNE']['A'], '4MNE interface, chain A'),
                legend_dot(CTRL_COLOR_REAL, '4MNE controls'),
                legend_dot(IFACE_SHADES['7K0V']['B'], '7K0V interface, C-lobe'),
                legend_dot(IFACE_SHADES['7K0V']['A'], '7K0V interface, N-lobe'),
                legend_dot(CTRL_COLOR_ALT, '7K0V controls')],
        caption=f'Both BRAF protomers coloured by {assay}; C-beta spheres mark every residue '
                'compared, on both protomers, each chain using its own interface and '
                'control sets.  4MNE is the wild-type BRAF:MEK1 complex with BRAF apo; '
                'only the side-to-side dimer (R509, left) is functionally distinct.')

show(FIG6 / 'braf_braf_activity_withspheres.png')
show(FIG6 / 'braf_braf_abundance_withspheres.png')

In [ ]:
# Stand-alone keys to place beside the panels: spheres, cartoons, colour scale.
for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['braf_cterm', 'braf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre, cap=0.85 if assay == 'activity' else None)
    plot_structure_key(
        FIG6 / f'braf_braf_{assay}_key', centre=centre, vmin=vmin, vmax=vmax,
        cbar_label=f'BRAF {assay} (missense mean)',
        spheres=[(IFACE_SHADES['4MNE']['B'], '4MNE interface, chain B', 0.85),
                 (IFACE_SHADES['4MNE']['A'], '4MNE interface, chain A', 0.85),
                 (CTRL_COLOR_REAL, '4MNE controls', 0.42),
                 (IFACE_SHADES['7K0V']['B'], '7K0V interface, C-lobe', 0.85),
                 (IFACE_SHADES['7K0V']['A'], '7K0V interface, N-lobe', 0.85),
                 (CTRL_COLOR_ALT, '7K0V controls', 0.42)],
        sticks='Interface residues')
show(FIG6 / 'braf_braf_activity_key.png', width=260)

### ED9d — score distributions by category

Brackets apply the rule of section B to each comparison (BH-FDR q < 0.05 from
the Mann–Whitney test, and |Cohen's d| ≥ 0.2), so a shift that is detectable but negligible in size is
marked "ns".

In [ ]:
# Each contact beside the control it was tested against, then the two contacts
# against each other. No bracket pools controls across structures.
plot_interface_violins(
    braf_pos, braf, protein='BRAF', cats=[REAL, RCTRL, ALT, ACTRL, ALLD], colors=CAT_COLOR,
    pairs=BRAF_PAIRS, q=braf_q,
    title='BRAF interface residues: side-to-side dimer (4MNE, BRAF:MEK1)\n'
          'vs. alternate contact (7K0V), each vs. its own controls',
    out=FIG6 / 'braf_braf_violins')
show(FIG6 / 'braf_braf_violins.png')

### CRAF on the CRAF–HSP90β–CDC37 complex (7Z38)

The CDC37 and HSP90β contacts of the cryo-EM structure shown in ED9e. Because
7Z38 is a ternary complex, both interfaces are tested against one shared control
set here: the union of the two complexes' control sets, minus both interfaces
(each complex's own control set includes some of the other chaperone's
interface). The q-values are BH across all complexes per assay, as in section B,
with these two tests in place of the complexes' own. Other CRAF–chaperone
structures are not pooled in.

In [ ]:
craf = results[(results['scored_protein_name'] == 'RAF1')
               & results['structure'].str.contains('7Z38')
               & results['interface_positions'].notna()]


def contact_sets(partner):
    """Interface and control positions of the 7Z38 complex with `partner`."""
    rows = craf[craf['interactor_name'] == partner]
    iface = set().union(*map(parse_positions, rows['interface_positions']))
    ctrl = set().union(*map(parse_positions, rows['control_positions'].dropna()))
    return iface, ctrl - iface


cdc37, cdc37_ctrl = contact_sets('CDC37')
hsp90, hsp90_ctrl = contact_sets('HSP90AB1')
craf_ctrl = (cdc37_ctrl | hsp90_ctrl) - cdc37 - hsp90      # shared (union) control
CDC37_I, HSP90_I, CRAF_CTRL = 'CDC37\ninterface', 'HSP90\ninterface', 'Control\n(union)'
craf_pos = {CDC37_I: cdc37, HSP90_I: hsp90, CRAF_CTRL: craf_ctrl}
CRAF_PAIRS = [(CDC37_I, CRAF_CTRL), (HSP90_I, CRAF_CTRL)]
# sphere colours: each interface a darker shade of its partner's colour
IFACE_DARK = {'CDC37': darken(PARTNER_COLOR['CDC37']), 'HSP90': darken(PARTNER_COLOR['HSP90'])}
craf_scores = library_scores(['craf_cterm', 'craf_nterm'])
craf_tests = interface_tests(craf_scores, craf_pos, CRAF_PAIRS)

# q: BH per assay over all complexes, with these two tests in place of the
# 7Z38 CDC37 and HSP90 complexes' own-control tests.
craf_q = {assay: substituted_q(
              complexes, assay,
              lambda r: ((r['scored_protein_name'] == 'RAF1') & r['structure'].str.contains('7Z38')
                         & r['interactor_name'].isin(['CDC37', 'HSP90AB1'])),
              {pair: craf_tests.loc[(assay, label(pair[0]), label(pair[1])), 'p']
               for pair in CRAF_PAIRS})
          for assay in ASSAYS}
craf_tests['q'] = [craf_q[a][next(pr for pr in CRAF_PAIRS if (label(pr[0]), label(pr[1])) == (i, c))]
                   for a, i, c in craf_tests.index]
craf_tests.assign(p=craf_tests['p'].map('{:.1e}'.format), q=craf_tests['q'].map('{:.1e}'.format))

### ED9e — CRAF coloured by mean missense score

CRAF (chain C) coloured per residue (the abundance render is the ED9e panel;
activity is drawn too), chaperones semi-transparent, interface residues as
ball-and-stick. Rendered with `render_structure`.

In [ ]:
# Spheres on CRAF: each interface in a darker shade of its partner's colour;
# controls in grey (the shared union control, craf_ctrl).
spheres = [('C', cdc37, IFACE_DARK['CDC37'], 0.62),
           ('C', hsp90, IFACE_DARK['HSP90'], 0.62),
           ('C', craf_ctrl, CTRL_COLOR_REAL, 0.42)]

for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['craf_cterm', 'craf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre)
    render = render_structure(
        'data/interactions/pdb_controls/_raw/7Z38.pdb', FIG6 / f'_craf_hsp90_cdc37_{assay}_render.png',
        scored_chains=['C'], scores=pos, centre=centre, vmin=vmin, vmax=vmax,
        partners={'A+B': PARTNER_COLOR['HSP90'], 'D': PARTNER_COLOR['CDC37']},
        sticks={'C': cdc37 | hsp90}, contacts_from='A+B+D', spheres=spheres,
        view=STRUCTURE_VIEWS['7Z38'], size=(6800, 2400), set_viewport=False)
    plot_structure_panel(
        [render], [f'RAF1 (CRAF) {assay} on the RAF1–HSP90–CDC37 complex (7Z38)'],
        FIG6 / f'craf_hsp90_cdc37_{assay}', layout='single',
        centre=centre, vmin=vmin, vmax=vmax, cbar_label=f'CRAF {assay}',
        legend=[legend_patch(PARTNER_COLOR['HSP90'], 'HSP90β (chains A, B)'),
                legend_patch(PARTNER_COLOR['CDC37'], 'CDC37 (chain D)'),
                legend_dot(IFACE_DARK['HSP90'], 'HSP90β interface'),
                legend_dot(IFACE_DARK['CDC37'], 'CDC37 interface'),
                legend_dot(CTRL_COLOR_REAL, 'Controls')],
        caption=f'RAF1 (chain C) opaque cartoon coloured by {assay}; chaperones '
                'semi-transparent; interface residues shown as ball-and-stick; '
                'C-beta spheres mark the interface and control positions compared.')
print(f'CRAF spheres: {len(cdc37)} CDC37 interface, {len(hsp90)} HSP90 interface, '
      f'{len(craf_ctrl)} control positions')

show(FIG6 / 'craf_hsp90_cdc37_abundance.png')
show(FIG6 / 'craf_hsp90_cdc37_activity.png')

In [ ]:
for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['craf_cterm', 'craf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre)
    plot_structure_key(
        FIG6 / f'craf_hsp90_cdc37_{assay}_key', centre=centre, vmin=vmin, vmax=vmax,
        cbar_label=f'CRAF {assay} (missense mean)',
        partners=[(PARTNER_COLOR['HSP90'], 'HSP90β (chains A, B)'),
                  (PARTNER_COLOR['CDC37'], 'CDC37 (chain D)')],
        spheres=[(IFACE_DARK['HSP90'], 'HSP90β interface', 0.62),
                 (IFACE_DARK['CDC37'], 'CDC37 interface', 0.62),
                 (CTRL_COLOR_REAL, 'Controls (union)', 0.42)],
        sticks='Interface residues, both sides')
show(FIG6 / 'craf_hsp90_cdc37_abundance_key.png', width=260)

### ED9f — CRAF chaperone-interface residues vs. their controls

7Z38 only: each interface next to the shared (union) control, then all
scored positions. Brackets apply the rule of section B (BH-FDR q < 0.05, and |Cohen's d|
≥ 0.2), as in ED9d.

In [ ]:
plot_interface_violins(
    craf_pos, craf_scores, protein='CRAF', cats=[CDC37_I, HSP90_I, CRAF_CTRL, ALLD],
    colors={CDC37_I: PARTNER_COLOR['CDC37'], HSP90_I: PARTNER_COLOR['HSP90'],
            CRAF_CTRL: CTRL_COLOR_REAL, ALLD: ALL_COLOR},
    pairs=CRAF_PAIRS,
    q=craf_q,
    title='CRAF interface residues on CRAF–HSP90–CDC37 (7Z38)\n'
          'each interface vs. the shared (union) control; BH-FDR q',
    out=FIG6 / 'craf_interface_violins')
show(FIG6 / 'craf_interface_violins.png')

### ED9g and ED9h — BRAF–ITCH, a predicted interface (AlphaFold-Multimer)

The E3 ubiquitin ligase ITCH, from the Predictomes screen. The structure panel
shows the model with the most interface positions (the rank-1 model), and the
interface and its own controls below come from that model only.

In [ ]:
itch_rows = results[(results['scored_protein_name'] == 'BRAF')
                    & (results['interactor_name'] == 'ITCH')
                    & results['interface_positions'].notna()]
shown = itch_rows.loc[itch_rows['n_interface_positions'].idxmax(), 'structure']
rows = itch_rows[itch_rows['structure'] == shown]
iface = set().union(*map(parse_positions, rows['interface_positions']))
ITCH_I, ITCH_CTRL = 'ITCH\ninterface', 'Control'
itch_pos = {ITCH_I: iface,
            ITCH_CTRL: set().union(*map(parse_positions, rows['control_positions'].dropna())) - iface}
print(shown)
itch_pdb = resolve_pdb_path(shown)
braf_chain = rows['scored_chain'].iloc[0]
itch_chain = next(ch for ch in plddt_keep(itch_pdb, 0) if ch != braf_chain)
ITCH_DARK = darken(PARTNER_COLOR['ITCH'])      # interface sphere colour
# q: this complex's own BH-FDR q-value (section B), per assay
itch_q = (complexes[(complexes['structure'] == shown) & (complexes['scored_protein_name'] == 'BRAF')]
          .set_index('assay')['pval_fdr'].to_dict())
itch_tests = interface_tests(braf, itch_pos, [(ITCH_I, ITCH_CTRL)])
itch_tests['q'] = [f'{itch_q[a]:.1e}' for a, _, _ in itch_tests.index]
itch_tests.assign(p=itch_tests['p'].map('{:.1e}'.format))

ED9g: BRAF (chain A) coloured per residue by mean missense WT-relative score, ITCH
as semi-transparent purple cartoon, and interface residues as ball-and-stick;
disordered regions (pLDDT < 70) are hidden. Rendered with `render_structure`.

In [ ]:
itch_ctrl = itch_pos[ITCH_CTRL]
# hide pLDDT < 70, but keep every position compared (interface and controls)
keep = plddt_keep(itch_pdb, 70, always={braf_chain: iface | itch_ctrl})
# Spheres on BRAF: interface in a darker shade of ITCH's colour, controls grey.
spheres = [(braf_chain, iface, ITCH_DARK, 0.62), (braf_chain, itch_ctrl, CTRL_COLOR_REAL, 0.42)]

for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['braf_cterm', 'braf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre, cap=0.85 if assay == 'activity' else None)
    render = render_structure(
        itch_pdb, FIG6 / f'_braf_itch_{assay}_render.png',
        scored_chains=[braf_chain], scores=pos, centre=centre, vmin=vmin, vmax=vmax,
        partners={itch_chain: PARTNER_COLOR['ITCH']}, sticks={braf_chain: iface},
        contacts_from=itch_chain, keep=keep, spheres=spheres,
        view=STRUCTURE_VIEWS['BRAF-ITCH'], size=(4200, 2400))
    plot_structure_panel(
        [render], [f'BRAF {assay} on the BRAF–ITCH complex (AlphaFold-Multimer)'],
        FIG6 / f'braf_itch_{assay}', layout='compact', width=10.0,
        centre=centre, vmin=vmin, vmax=vmax, cbar_label=f'BRAF {assay}',
        legend=[legend_patch(PARTNER_COLOR['ITCH'], f'ITCH (chain {itch_chain})'),
                legend_dot(ITCH_DARK, 'ITCH interface'), legend_dot(CTRL_COLOR_REAL, 'Controls')],
        caption=f'BRAF (chain {braf_chain}) opaque cartoon coloured by {assay}; ITCH shown as '
                'semi-transparent cartoon; C-beta spheres mark the interface and control positions '
                'compared.  Other disordered (pLDDT < 70) regions hidden.')

show(FIG6 / 'braf_itch_activity.png')
show(FIG6 / 'braf_itch_abundance.png')

In [ ]:
for assay in ['activity', 'abundance']:
    pos, centre = residue_scores(scores, ['braf_cterm', 'braf_nterm'], assay)
    vmin, vmax = score_scale(pos, centre, cap=0.85 if assay == 'activity' else None)
    plot_structure_key(
        FIG6 / f'braf_itch_{assay}_key', centre=centre, vmin=vmin, vmax=vmax,
        cbar_label=f'BRAF {assay} (missense mean)',
        partners=[(PARTNER_COLOR['ITCH'], f'ITCH (chain {itch_chain})')],
        spheres=[(ITCH_DARK, 'ITCH interface', 0.62), (CTRL_COLOR_REAL, 'Controls', 0.42)],
        sticks='Interface residues, both sides')
show(FIG6 / 'braf_itch_activity_key.png', width=260)

ED9h: score distributions for the ITCH interface, its controls, and all domain
positions; brackets as in ED9d.

In [ ]:
plot_interface_violins(
    itch_pos, braf, protein='BRAF', cats=[ITCH_I, ITCH_CTRL, ALLD],
    colors={ITCH_I: PARTNER_COLOR['ITCH'], ITCH_CTRL: '#9e9e9e', ALLD: ALL_COLOR},
    pairs=[(ITCH_I, ITCH_CTRL)], q={a: {(ITCH_I, ITCH_CTRL): itch_q[a]} for a in ASSAYS},
    title='BRAF residue-category score distributions\n(ITCH interface vs. control vs. all)',
    out=FIG6 / 'braf_itch_violins')
show(FIG6 / 'braf_itch_violins.png')

## E. Where the partners bind: interface clusters

Each partner's **footprint** is the union of its interface residues on the
scored protein over every assessable complex — regardless of significance, so the
clustering does not depend on the functional result. Partners with fewer than 5
scored interface residues are left out. Two footprints are compared by
intersection-over-union, with a 6 Å Cβ tolerance so that a site engaged one
residue off-register still counts as shared, and partners are clustered by
complete linkage on 1 − IoU, cut at IoU ≥ 0.5: every pair of partners in a
cluster shares at least half its footprint.

In [ ]:
valid = results[results['interface_positions'].notna()]
footprints = build_footprints(valid)                # {protein: {partner: residues}}
sig_pairs = set(zip(complexes.loc[complexes['supported'], 'scored_protein_name'],
                    complexes.loc[complexes['supported'], 'interactor']))   # supported (protein, partner)
partner_name = dict(zip(valid['interactor'], valid['interactor_name']))
coords = representative_cbeta_coords(valid)         # for the 6 A tolerance

sites = cluster_sites(footprints, sig_pairs, coords, SITE_CUTOFF, DIST_TOL, MIN_FOOTPRINT)


def site_table(sites, cutoff):
    return pd.DataFrame([{
        'iou_cutoff': cutoff,
        'protein': disp(s['protein']),
        'n_partners': s['n_partners'],
        'n_supported_partners': s['n_supported'],
        'partner_category': site_category_2(s['n_partners']),
        'range': s['range'],
        'core_residues': ','.join(map(str, s['core_residues'])),
        # partner gene names; '*' marks functionally supported partners
        'partner_names': ';'.join(
            disp(partner_name.get(x, x)) + ('*' if (s['protein'], x) in sig_pairs else '')
            for x in sorted(s['partners'], key=lambda u: disp(partner_name.get(u, u)))),
        'partners': ';'.join(sorted(s['partners'])),
    } for s in sites])


clusters = site_table(sites, SITE_CUTOFF)
interactions = clusters['n_partners'].sum()
at_shared = clusters.loc[clusters['n_partners'] >= 2, 'n_partners'].sum()
print(f"{len(clusters)} interface clusters from {interactions} partner interfaces; "
      f"{at_shared} ({100 * at_shared / interactions:.0f}%) sit in a cluster shared "
      f"with at least one other partner")
clusters.groupby('partner_category').agg(clusters=('protein', 'size'),
                                         partners=('n_partners', 'sum'))

### ED9i — interface clusters with one partner or several

Clusters engaged by a single partner, or shared by two or more (promiscuous).
ED9i is the small bar for all proteins and KRAS (as percentages); the
per-protein bar drawn first is not in the paper and is kept for reference.
The table below repeats the count at looser and tighter cutoffs;
`interacting_sites.csv` lists every cluster with its core residues (contacted by
at least half its partners) and partners.

In [ ]:
plot_sites_bar(sites, FIG6 / 'interacting_sites_by_partner_count_iou0p5.pdf',
               SITE_CUTOFF, MIN_FOOTPRINT,
               categories=(site_category_2, SITE_CATS_2, SITE_CAT_COLOR_2, SITE_CAT_DESC_2))
show(FIG6 / 'interacting_sites_by_partner_count_iou0p5.png')

# The same split for all proteins and for KRAS alone, small enough to sit beside
# the KRAS clustering: ED9i.
plot_sites_bar_compact(sites, FIG6 / 'site_heatmaps/sites_by_partner_count_all_vs_KRAS',
                       ['KRAS'], categories=(site_category_2, SITE_CATS_2,
                                             SITE_CAT_COLOR_2, SITE_CAT_DESC_2))
show(FIG6 / 'site_heatmaps/sites_by_partner_count_all_vs_KRAS.png')

by_cutoff = pd.concat([site_table(cluster_sites(footprints, sig_pairs, coords, cut,
                                                DIST_TOL, MIN_FOOTPRINT), cut)
                       for cut in [0.2, 0.3, 0.4, 0.5]])
by_cutoff.to_csv(FIG6 / 'interacting_sites.csv', index=False)
by_cutoff.groupby(['iou_cutoff', 'partner_category']).size().unstack()

### ED9j — partner-interface overlap for KRAS and BRAF

Pairwise footprint IoU between each protein's partners, ordered by clustering,
with the IoU ≥ 0.5 clusters boxed; the strip at right is each partner's median
missense Cohen's d per assay over its complexes, grey unless at least one of its
complexes has q < 0.05; hub clusters carry family labels.

In [ ]:
# per (protein, partner, assay): median Cohen's d over the partner's complexes,
# and the smallest q among them
effects = {key: (float(g['cohens_d'].median()), float(g['pval_fdr'].min()))
           for key, g in complexes.groupby(['scored_protein_name', 'interactor', 'assay'])}
panels = []
for protein in ['KRAS', 'BRAF']:
    items = [(x, f) for x, f in footprints[protein].items() if len(f) >= MIN_FOOTPRINT]
    items.sort(key=lambda t: disp(partner_name.get(t[0], t[0])).lower())
    panels.append((protein, items))

plot_iou_heatmaps(panels, coords, footprints, sig_pairs, partner_name, effects)
show(FIG6 / 'site_heatmaps/KRAS_BRAF_sites_iou0p5_paper_leaders_combined.png')
clusters[clusters['protein'].isin(['KRAS', 'BRAF'])
         & (clusters['n_partners'] >= 5)][['protein', 'n_partners', 'core_residues']]

### ED9k — the KRAS surface painted by interface cluster

Three views (±90° about the vertical axis). A residue belongs to a family if at
least half that family's partners contact it; residues shared by two or more
families are shaded by how many. Rendered with `render_surface`.

In [ ]:
# The KRAS interface families are the labelled hub clusters of ED9j.
families = _paper_panel_data('KRAS', panels[0][1], coords, partner_name, SITE_CUTOFF)[3]
rep = select_representative_structures(valid)['KRAS']
footprint_residues = set().union(*(footprints['KRAS'][u] for fam in families for u in fam['members']))
drawn = surface_residues(coords['KRAS'], rep['pdb_path'], rep['chain'], rep['predictor'],
                         keep_always=footprint_residues)

# A residue belongs to a family if at least half of that family's partners
# contact it; it can belong to several.
membership = defaultdict(list)
for fam in families:
    contacts = Counter(r for u in fam['members'] for r in footprints['KRAS'][u])
    for r, n in contacts.items():
        if r in drawn and n / fam['n'] >= 0.5:
            membership[r].append(fam['num'])

# One family: that family's colour. Two or more: a purple shade by how many.
paint = defaultdict(list)
for r, fams in membership.items():
    colour = (FAMILY_PALETTE[(fams[0] - 1) % len(FAMILY_PALETTE)] if len(fams) == 1
              else shared_tier(len(fams))[0])
    paint[colour].append(r)
n_shared = sum(len(f) > 1 for f in membership.values())
print(f'{len(membership)} core residues, {n_shared} shared by two or more families')

yaws = [-90, 0, 90]
imgs = render_surface(rep['pdb_path'], rep['chain'], drawn, dict(paint),
                      FIG6 / 'site_heatmaps' / '_kras_render', 'KRAS_interface_residues',
                      view=STRUCTURE_VIEWS['KRAS-surface'], yaws=yaws)
alone = {f[0] for f in membership.values() if len(f) == 1}
legend = [legend_dot(FAMILY_PALETTE[(fam['num'] - 1) % len(FAMILY_PALETTE)], fam['label'], edge='white', size=10)
          for fam in sorted(families, key=lambda f: f['num']) if fam['num'] in alone]
legend += [legend_dot(col, f'Shared: {lab}', edge='white', size=10) for lo, hi, col, lab in SHARED_TIERS
           if any(lo <= len(f) <= hi for f in membership.values())]
plot_surface_views(
    imgs, ['KRAS (0°)' if y == 0 else f'{y:+d}° (y)' for y in yaws],
    FIG6 / 'site_heatmaps' / 'KRAS_interface_residues', legend=legend,
    title=f'KRAS interface site groups painted on the structure (IoU ≥ {SITE_CUTOFF:g})',
    caption='A residue belongs to a family if contacted by ≥ 50% of that family’s interactors; '
            'residues in one family are coloured by it, those in ≥ 2 families are shaded by how '
            f'many.  {n_shared} of {len(membership)} core residues belong to ≥ 2 families.')

show(FIG6 / 'site_heatmaps/KRAS_interface_residues.png')